# 01 · ANOVA de una vía (Python)

**Semana 1 — Fundamentos y un solo factor.**

## Objetivos
- Ajustar un ANOVA de una vía para un experimento de un solo factor.
- Verificar los supuestos mediante el análisis de residuales.
- Comparar tratamientos con **Tukey HSD** y, frente a un control, con **Dunnett**.

## Paquetes
`pandas`, `numpy`, `matplotlib`, `scipy`, `statsmodels`.

> Teoría: [`../teoria/03-anova-una-via.md`](../teoria/03-anova-una-via.md) y [`../teoria/04-comparaciones-multiples.md`](../teoria/04-comparaciones-multiples.md).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd

np.random.seed(1)  # reproducibilidad
pd.set_option('display.precision', 4)

## 1. Los datos

Resistencia a la tensión (lb/pulg²) de una fibra sintética según el **porcentaje de algodón** (factor con 5 niveles: 15, 20, 25, 30, 35 %), 5 réplicas por nivel (Montgomery, ej. 3.1).

In [ ]:
df = pd.read_csv('../datos/resistencia-algodon.csv')
df['algodon_pct'] = df['algodon_pct'].astype('category')
df.groupby('algodon_pct', observed=True)['resistencia'].describe()[['count','mean','std']]

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
df.boxplot(column='resistencia', by='algodon_pct', ax=ax, grid=False)
ax.set_title('Resistencia por % de algodón'); plt.suptitle('')
ax.set_xlabel('% algodón'); ax.set_ylabel('Resistencia'); plt.show()

La dispersión sugiere que la media crece hasta ~30 % y luego baja. Lo contrastamos formalmente con el ANOVA.

## 2. ANOVA de una vía

Modelo de efectos: $y_{ij} = \mu + \tau_i + \varepsilon_{ij}$, con $H_0: \tau_1=\cdots=\tau_5=0$.

In [ ]:
modelo = ols('resistencia ~ C(algodon_pct)', data=df).fit()
tabla = sm.stats.anova_lm(modelo, typ=2)
print(tabla)
print(f'\nR2 = {modelo.rsquared:.4f}')

Valor-p $\approx 9\times10^{-6}$: **se rechaza $H_0$**. El % de algodón afecta la resistencia y explica $R^2\approx 75\%$ de la variabilidad. Antes de comparar medias, validamos los supuestos.

## 3. Verificación de supuestos (residuales)

In [ ]:
resid = modelo.resid
ajust = modelo.fittedvalues
fig, ax = plt.subplots(1, 2, figsize=(11,4))
sm.qqplot(resid, line='s', ax=ax[0])
ax[0].set_title('Q-Q normal de residuales')
ax[1].scatter(ajust, resid)
ax[1].axhline(0, color='gray', ls='--')
ax[1].set_xlabel('Valores ajustados'); ax[1].set_ylabel('Residual')
ax[1].set_title('Residuales vs. ajustados')
plt.tight_layout(); plt.show()

In [ ]:
print('Shapiro-Wilk (normalidad):', stats.shapiro(resid))
grupos = [g['resistencia'].values for _, g in df.groupby('algodon_pct', observed=True)]
print('Levene (homocedasticidad):', stats.levene(*grupos))

Shapiro-Wilk ($p\approx0.18$) y Levene ($p\approx0.86$) no rechazan: no hay evidencia contra normalidad ni varianzas iguales. Los supuestos se sostienen.

## 4. Comparaciones múltiples: Tukey HSD

Tukey controla el error por familia para **todas las parejas**.

In [ ]:
tukey = pairwise_tukeyhsd(df['resistencia'], df['algodon_pct'], alpha=0.05)
print(tukey.summary())

In [ ]:
tukey.plot_simultaneous()
plt.xlabel('Resistencia media'); plt.show()

El nivel **30 %** da la mayor resistencia y supera a 15, 20 y 35 %; 15 % y 35 % no difieren entre sí.

## 5. Comparaciones contra un control: Dunnett

Tiempos de coagulación según la **dieta** (A = control; B, C, D = tratamientos). Dunnett es más potente que Tukey cuando solo interesa *tratamiento vs. control*.

In [ ]:
coag = pd.read_csv('../datos/tiempo-coagulacion.csv')
grupos_c = {k: g['tiempo'].values for k, g in coag.groupby('dieta')}
res = stats.dunnett(grupos_c['B'], grupos_c['C'], grupos_c['D'],
                    control=grupos_c['A'])
pd.DataFrame({'comparacion': ['B-A','C-A','D-A'],
              'estadistico': res.statistic,
              'p_value': res.pvalue})

Frente al control A, **B y C** difieren significativamente ($p<0.05$); **D** no.

## 6. Conclusiones

- El % de algodón **afecta** la resistencia (ANOVA, $p\approx 9\times10^{-6}$, $R^2\approx0.75$).
- Supuestos verificados (normalidad y homocedasticidad).
- **Tukey:** 30 % maximiza la resistencia.
- **Dunnett:** B y C superan al control A; D no.

> Los resultados deben coincidir con la versión en R ([`01-anova-una-via_r.ipynb`](01-anova-una-via_r.ipynb)).

---

## Ejemplo 2 – Diferente dominio: rendimiento de cultivo por fertilizante

Rendimiento (kg/ha) bajo **cuatro fertilizantes** (A–D), 5 réplicas cada uno. Mismo flujo ANOVA aplicado a un contexto agrícola.

In [ ]:
fert = pd.DataFrame({
    'fertilizante': np.repeat(['A','B','C','D'], 5),
    'rendimiento':  [18, 20, 19, 22, 21,
                     25, 27, 26, 28, 24,
                     22, 23, 21, 24, 23,
                     16, 17, 15, 18, 16]
})
fig, ax = plt.subplots(figsize=(6, 4))
fert.boxplot(column='rendimiento', by='fertilizante', ax=ax, grid=False)
ax.set_title('Rendimiento por fertilizante'); plt.suptitle('')
ax.set_xlabel('Fertilizante'); ax.set_ylabel('Rendimiento (kg/ha)'); plt.show()
fert.groupby('fertilizante')['rendimiento'].describe()[['count','mean','std']]

In [ ]:
mod_f = ols('rendimiento ~ C(fertilizante)', data=fert).fit()
print(sm.stats.anova_lm(mod_f, typ=2))
print(f'R2 = {mod_f.rsquared:.4f}')

In [ ]:
resid_f = mod_f.resid; ajust_f = mod_f.fittedvalues
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sm.qqplot(resid_f, line='s', ax=ax[0]); ax[0].set_title('Q-Q normal de residuales')
ax[1].scatter(ajust_f, resid_f); ax[1].axhline(0, color='gray', ls='--')
ax[1].set_xlabel('Ajustados'); ax[1].set_ylabel('Residual')
ax[1].set_title('Residuales vs. ajustados')
plt.tight_layout(); plt.show()
print('Shapiro-Wilk:', stats.shapiro(resid_f))
grupos_f = [g['rendimiento'].values for _, g in fert.groupby('fertilizante')]
print('Levene:', stats.levene(*grupos_f))

Supuestos verificados ($p$-Shapiro $>0.05$, $p$-Levene $>0.05$). Procedemos con Tukey HSD.

In [ ]:
tukey_f = pairwise_tukeyhsd(fert['rendimiento'], fert['fertilizante'], alpha=0.05)
print(tukey_f.summary())
tukey_f.plot_simultaneous()
plt.xlabel('Rendimiento medio (kg/ha)'); plt.show()

**B** supera significativamente a todos los demás fertilizantes; **D** da el menor rendimiento y difiere de A, B y C.

---

## Ejemplo 3 – Supuestos incumplidos: defectos por operario

Número de defectos diarios producidos por **cinco operarios** (5 días cada uno). Los datos de conteo son típicamente asimétricos y heterocedásticos: un caso donde los supuestos del ANOVA fallan.

In [ ]:
defectos = pd.DataFrame({
    'operario':    np.repeat([f'Op{i}' for i in range(1, 6)], 5),
    'n_defectos':  [ 1,  2,  1,  2,  1,
                     4,  8,  5, 12,  6,
                    20, 35, 25, 45, 30,
                     2,  1,  3,  2,  1,
                     8, 15, 10, 20, 12]
})
fig, ax = plt.subplots(figsize=(6, 4))
defectos.boxplot(column='n_defectos', by='operario', ax=ax, grid=False)
ax.set_title('Defectos por operario'); plt.suptitle('')
ax.set_xlabel('Operario'); ax.set_ylabel('Defectos diarios'); plt.show()
defectos.groupby('operario')['n_defectos'].describe()[['count','mean','std']]

In [ ]:
mod_d = ols('n_defectos ~ C(operario)', data=defectos).fit()
print(sm.stats.anova_lm(mod_d, typ=2))
resid_d = mod_d.resid; ajust_d = mod_d.fittedvalues
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sm.qqplot(resid_d, line='s', ax=ax[0]); ax[0].set_title('Q-Q normal de residuales')
ax[1].scatter(ajust_d, resid_d); ax[1].axhline(0, color='gray', ls='--')
ax[1].set_xlabel('Ajustados'); ax[1].set_ylabel('Residual')
ax[1].set_title('Residuales vs. ajustados')
plt.tight_layout(); plt.show()
print('Shapiro-Wilk:', stats.shapiro(resid_d))
grupos_d = [g['n_defectos'].values for _, g in defectos.groupby('operario')]
print('Levene:', stats.levene(*grupos_d))

Levene ($p < 0.05$): **varianzas heterogéneas**; Shapiro-Wilk rechaza normalidad. El ANOVA clásico **no es apropiado** para estos datos. Alternativas:

- **Transformación** $\sqrt{y}$ o $\log(y+1)$ para estabilizar la varianza.
- **Kruskal-Wallis**: prueba no paramétrica equivalente (ver `02-no-parametricas-transformaciones`).

In [ ]:
stat_kw, p_kw = stats.kruskal(*grupos_d)
print(f'Kruskal-Wallis: H = {stat_kw:.4f},  p = {p_kw:.4f}')

Kruskal-Wallis confirma diferencias significativas entre operarios sin asumir normalidad ni homocedasticidad.

---

## Ejemplo 4 – Dataset real: `PlantGrowth`

Dataset `PlantGrowth` (disponible vía `statsmodels`): peso seco de plantas bajo un **control** y dos **tratamientos** (trt1, trt2), 10 réplicas por grupo (Dobson, 1983). Ilustra que un ANOVA global significativo no implica que *todas* las comparaciones por pares sean significativas.

In [ ]:
try:
    plants = sm.datasets.get_rdataset('PlantGrowth', 'datasets', cache=True).data
except Exception:
    plants = pd.DataFrame({
        'weight': [
            4.17, 5.58, 5.18, 6.11, 4.50, 4.61, 5.17, 4.53, 5.33, 5.14,  # ctrl
            4.81, 4.17, 4.41, 3.59, 5.87, 3.83, 6.03, 4.89, 4.32, 4.69,  # trt1
            6.31, 5.12, 5.54, 5.50, 5.37, 5.29, 4.92, 6.15, 5.80, 5.26   # trt2
        ],
        'group': ['ctrl'] * 10 + ['trt1'] * 10 + ['trt2'] * 10
    })

plants['group'] = plants['group'].astype('category')
fig, ax = plt.subplots(figsize=(6, 4))
plants.boxplot(column='weight', by='group', ax=ax, grid=False)
ax.set_title('PlantGrowth: peso por tratamiento'); plt.suptitle('')
ax.set_xlabel('Grupo'); ax.set_ylabel('Peso seco (g)'); plt.show()
plants.groupby('group')['weight'].describe()[['count','mean','std']]

In [ ]:
mod_pg = ols('weight ~ C(group)', data=plants).fit()
print(sm.stats.anova_lm(mod_pg, typ=2))
print(f'R2 = {mod_pg.rsquared:.4f}')

In [ ]:
resid_pg = mod_pg.resid; ajust_pg = mod_pg.fittedvalues
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sm.qqplot(resid_pg, line='s', ax=ax[0]); ax[0].set_title('Q-Q normal de residuales')
ax[1].scatter(ajust_pg, resid_pg); ax[1].axhline(0, color='gray', ls='--')
ax[1].set_xlabel('Ajustados'); ax[1].set_ylabel('Residual')
ax[1].set_title('Residuales vs. ajustados')
plt.tight_layout(); plt.show()
print('Shapiro-Wilk:', stats.shapiro(resid_pg))
grupos_pg = [g['weight'].values for _, g in plants.groupby('group')]
print('Levene:', stats.levene(*grupos_pg))

In [ ]:
tukey_pg = pairwise_tukeyhsd(plants['weight'], plants['group'], alpha=0.05)
print(tukey_pg.summary())
tukey_pg.plot_simultaneous()
plt.xlabel('Peso medio (g)'); plt.show()

Solo **trt1 vs. trt2** resulta significativo ($p\approx0.01$); trt2–ctrl roza la significancia ($p\approx0.09$). Lección: el $p$-valor global del ANOVA no garantiza que todas las comparaciones por pares lo sean.